# Tutorial 07: Bell State Similarity (BSS) — Directed Morphisms

This tutorial introduces the `bss` module, which provides a **directed similarity metric** for testing categorical morphisms between HLLSets.

## What You'll Learn

1. **BSS Definition** — The (τ, ρ) pair: inclusion and exclusion scores
2. **Asymmetry** — Why BSS(A→B) ≠ BSS(B→A)
3. **Morphism Testing** — When does a morphism A →(τ,ρ) B exist?
4. **BSS Matrix** — Pairwise similarity for collections
5. **Morphism Graphs** — Categorical structure discovery

## The BSS Formula

For sets A (source/query) and B (target/reference):

$$\text{BSS}_\tau(A \to B) = \frac{|A \cap B|}{|B|}$$

$$\text{BSS}_\rho(A \to B) = \frac{|A \setminus B|}{|B|}$$

Where:
- **τ (tau)** = inclusion score — "how much of B is covered by A?"
- **ρ (rho)** = exclusion score — "how much extraneous content does A have?"

### Key Properties

1. $0 \leq \tau \leq 1$, $0 \leq \rho$ (ρ can exceed 1 if |A| >> |B|)
2. When A ⊆ B: τ = |A|/|B|, ρ = 0
3. When A = B: τ = 1, ρ = 0 (identity morphism)
4. BSS is **NOT symmetric**: BSS(A→B) ≠ BSS(B→A)

In [1]:
# Import modules
import sys
sys.path.insert(0, '..')

from core.bss import (
    bss,
    bss_symmetric,
    test_morphism,
    bss_matrix,
    morphism_graph,
    bss_summary,
    BSSPair,
    MorphismResult,
)
from core.hllset import HLLSet

## 1. Basic BSS Computation

The `bss()` function computes the directed similarity from source A to target B.

In [2]:
# Create two HLLSets with some overlap
A = HLLSet.from_batch(["machine", "learning", "deep", "neural"])
B = HLLSet.from_batch(["machine", "learning", "algorithms", "models"])

print(f"Set A: ['machine', 'learning', 'deep', 'neural']")
print(f"Set B: ['machine', 'learning', 'algorithms', 'models']")
print(f"\n|A| ≈ {A.cardinality():.0f}")
print(f"|B| ≈ {B.cardinality():.0f}")
print(f"|A ∩ B| ≈ {A.intersect(B).cardinality():.0f}  (overlap: 'machine', 'learning')")

Set A: ['machine', 'learning', 'deep', 'neural']
Set B: ['machine', 'learning', 'algorithms', 'models']

|A| ≈ 5
|B| ≈ 6
|A ∩ B| ≈ 3  (overlap: 'machine', 'learning')


In [3]:
# Compute BSS(A → B)
pair = bss(A, B)

print(f"BSS(A → B):")
print(f"  τ (inclusion): {pair.tau:.4f}  — {pair.tau*100:.1f}% of B covered by A")
print(f"  ρ (exclusion): {pair.rho:.4f}  — extraneous A content relative to B")

BSS(A → B):
  τ (inclusion): 0.5000  — 50.0% of B covered by A
  ρ (exclusion): 0.5000  — extraneous A content relative to B


In [4]:
# BSS is NOT symmetric!
pair_BA = bss(B, A)

print(f"BSS(A → B): τ={pair.tau:.4f}, ρ={pair.rho:.4f}")
print(f"BSS(B → A): τ={pair_BA.tau:.4f}, ρ={pair_BA.rho:.4f}")
print(f"\nAsymmetry: BSS(A→B) ≠ BSS(B→A)")

BSS(A → B): τ=0.5000, ρ=0.5000
BSS(B → A): τ=0.6000, ρ=0.6000

Asymmetry: BSS(A→B) ≠ BSS(B→A)


## 2. Understanding the (τ, ρ) Space

The BSS pair gives a complete picture of the directed relationship:

| Situation | τ | ρ | Interpretation |
|-----------|---|---|----------------|
| A ⊆ B | |A|/|B| | 0 | A is a subset of B |
| A = B | 1 | 0 | Identity morphism |
| A ∩ B = ∅ | 0 | |A|/|B| | Disjoint sets |
| A covers B fully | 1 | (|A|-|B|)/|B| | A contains all of B plus extra |

In [5]:
# Case 1: Subset relationship (A ⊆ B)
A_sub = HLLSet.from_batch(["a", "b"])
B_super = HLLSet.from_batch(["a", "b", "c", "d"])

pair_subset = bss(A_sub, B_super)
print(f"A ⊆ B case:")
print(f"  A = {{'a', 'b'}}, B = {{'a', 'b', 'c', 'd'}}")
print(f"  BSS(A → B): τ={pair_subset.tau:.4f}, ρ={pair_subset.rho:.4f}")
print(f"  (A covers {pair_subset.tau*100:.0f}% of B, no extraneous content)")

A ⊆ B case:
  A = {'a', 'b'}, B = {'a', 'b', 'c', 'd'}
  BSS(A → B): τ=0.4000, ρ=0.0000
  (A covers 40% of B, no extraneous content)


In [6]:
# Case 2: Identity (A = B)
A_same = HLLSet.from_batch(["x", "y", "z"])
B_same = HLLSet.from_batch(["x", "y", "z"])

pair_identity = bss(A_same, B_same)
print(f"A = B case:")
print(f"  BSS(A → B): τ={pair_identity.tau:.4f}, ρ={pair_identity.rho:.4f}")
print(f"  (Perfect identity morphism)")

A = B case:
  BSS(A → B): τ=1.0000, ρ=0.0000
  (Perfect identity morphism)


In [7]:
# Case 3: Disjoint sets
A_dis = HLLSet.from_batch(["alpha", "beta"])
B_dis = HLLSet.from_batch(["gamma", "delta"])

pair_disjoint = bss(A_dis, B_dis)
print(f"Disjoint case:")
print(f"  A = {{'alpha', 'beta'}}, B = {{'gamma', 'delta'}}")
print(f"  BSS(A → B): τ={pair_disjoint.tau:.4f}, ρ={pair_disjoint.rho:.4f}")
print(f"  (No overlap, all A is extraneous to B)")

Disjoint case:
  A = {'alpha', 'beta'}, B = {'gamma', 'delta'}
  BSS(A → B): τ=0.0000, ρ=1.0000
  (No overlap, all A is extraneous to B)


In [8]:
# Case 4: A covers B completely plus extra
A_super = HLLSet.from_batch(["p", "q", "r", "s", "t", "u"])
B_sub = HLLSet.from_batch(["p", "q", "r"])

pair_superset = bss(A_super, B_sub)
print(f"A ⊃ B case:")
print(f"  A = {{'p','q','r','s','t','u'}}, B = {{'p','q','r'}}")
print(f"  BSS(A → B): τ={pair_superset.tau:.4f}, ρ={pair_superset.rho:.4f}")
print(f"  (A covers 100% of B, but has {pair_superset.rho*100:.0f}% extraneous)")

A ⊃ B case:
  A = {'p','q','r','s','t','u'}, B = {'p','q','r'}
  BSS(A → B): τ=1.0000, ρ=1.0000
  (A covers 100% of B, but has 100% extraneous)


## 3. Morphism Testing

A morphism A →(τ,ρ) B exists if and only if:

$$\text{BSS}_\tau(A \to B) \geq \tau_{threshold}$$
$$\text{BSS}_\rho(A \to B) \leq \rho_{threshold}$$

This formalizes "A covers B well enough, without too much noise."

In [9]:
# Create sets for morphism testing
source = HLLSet.from_batch(["machine", "learning", "deep", "neural", "networks"])
target = HLLSet.from_batch(["machine", "learning", "algorithms"])

# Test for morphism with default thresholds (τ≥0.7, ρ≤0.3)
result = test_morphism(source, target)

print(f"Morphism test: source → target")
print(f"  Result: {result}")
print(f"  Exists: {result.exists}")
print(f"  τ = {result.tau:.4f} (threshold: {result.tau_threshold})")
print(f"  ρ = {result.rho:.4f} (threshold: {result.rho_threshold})")

Morphism test: source → target
  Result: Morphism(↛, τ=0.6000≥0.7, ρ=0.8000≤0.3)
  Exists: False
  τ = 0.6000 (threshold: 0.7)
  ρ = 0.8000 (threshold: 0.3)


In [10]:
# Test with stricter thresholds
strict = test_morphism(source, target, tau_threshold=0.9, rho_threshold=0.1)

print(f"Stricter morphism test:")
print(f"  Result: {strict}")
print(f"  Margin τ: {strict.margin_tau:+.4f} (positive = passes τ test)")
print(f"  Margin ρ: {strict.margin_rho:+.4f} (positive = passes ρ test)")

Stricter morphism test:
  Result: Morphism(↛, τ=0.6000≥0.9, ρ=0.8000≤0.1)
  Margin τ: -0.3000 (positive = passes τ test)
  Margin ρ: -0.7000 (positive = passes ρ test)


In [11]:
# Test with relaxed thresholds
relaxed = test_morphism(source, target, tau_threshold=0.5, rho_threshold=0.8)

print(f"Relaxed morphism test:")
print(f"  Result: {relaxed}")
print(f"  Exists: {relaxed.exists}")

ValueError: Must have 0 ≤ ρ < τ ≤ 1, got ρ=0.8, τ=0.5

In [12]:
# MorphismResult contains full diagnostics
print(f"Diagnostics:")
print(f"  |source| ≈ {result.source_card:.0f}")
print(f"  |target| ≈ {result.target_card:.0f}")
print(f"  |source ∩ target| ≈ {result.intersection_card:.0f}")
print(f"  |source \\ target| ≈ {result.difference_card:.0f}")

Diagnostics:
  |source| ≈ 6
  |target| ≈ 5
  |source ∩ target| ≈ 3
  |source \ target| ≈ 4


## 4. BSS Summary — Human-Readable Analysis

The `bss_summary()` function provides a human-readable interpretation.

In [13]:
# Strong inclusion case
A_strong = HLLSet.from_batch(["a", "b", "c", "d", "e"])
B_strong = HLLSet.from_batch(["a", "b", "c", "d"])

print(bss_summary(A_strong, B_strong))

BSS(A → B):
  |A| ≈ 6.0,  |B| ≈ 5.0,  |A∩B| ≈ 5.0
  τ (inclusion) = 1.0000
  ρ (exclusion) = 0.4000
  τ + ρ         = 1.4000


In [14]:
# Disjoint case
A_disjoint = HLLSet.from_batch(["x", "y", "z"])
B_disjoint = HLLSet.from_batch(["p", "q", "r"])

print(bss_summary(A_disjoint, B_disjoint))

BSS(A → B):
  |A| ≈ 4.0,  |B| ≈ 4.0,  |A∩B| ≈ 0.0
  τ (inclusion) = 0.0000
  ρ (exclusion) = 1.0000
  τ + ρ         = 1.0000
  → A and B are nearly disjoint


In [15]:
# Subset case
A_subset = HLLSet.from_batch(["m", "n"])
B_superset = HLLSet.from_batch(["m", "n", "o", "p", "q"])

print(bss_summary(A_subset, B_superset))

BSS(A → B):
  |A| ≈ 3.0,  |B| ≈ 5.0,  |A∩B| ≈ 3.0
  τ (inclusion) = 0.6000
  ρ (exclusion) = 0.0000
  τ + ρ         = 0.6000
  → A ⊆ B (up to estimation error)


## 5. Symmetric BSS

For convenience, `bss_symmetric()` computes both directions at once.

In [16]:
A = HLLSet.from_batch(["deep", "learning", "neural"])
B = HLLSet.from_batch(["machine", "learning", "models"])

a_to_b, b_to_a = bss_symmetric(A, B)

print(f"Symmetric BSS:")
print(f"  A → B: τ={a_to_b.tau:.4f}, ρ={a_to_b.rho:.4f}")
print(f"  B → A: τ={b_to_a.tau:.4f}, ρ={b_to_a.rho:.4f}")
print(f"\nInterpretation:")
print(f"  A covers {a_to_b.tau*100:.0f}% of B")
print(f"  B covers {b_to_a.tau*100:.0f}% of A")

Symmetric BSS:
  A → B: τ=0.5000, ρ=0.7500
  B → A: τ=0.5000, ρ=0.7500

Interpretation:
  A covers 50% of B
  B covers 50% of A


## 6. BSS Matrix — Pairwise Analysis

For a collection of HLLSets, compute the pairwise BSS matrix.

In [17]:
# Create a collection of related sets
sets = [
    HLLSet.from_batch(["machine", "learning", "deep"]),
    HLLSet.from_batch(["machine", "learning", "algorithms"]),
    HLLSet.from_batch(["deep", "neural", "networks"]),
    HLLSet.from_batch(["natural", "language", "processing"]),
]

labels = ["ML-Deep", "ML-Algo", "DNN", "NLP"]

# Compute BSS matrix
matrix = bss_matrix(sets, labels=labels)

print(f"τ-matrix (inclusion scores):")
print(f"Entry (i,j) = fraction of set j covered by set i")
print()

τ-matrix (inclusion scores):
Entry (i,j) = fraction of set j covered by set i



In [18]:
# Display τ-matrix
import numpy as np

tau_mat = matrix['tau_matrix']

# Header
header = "         " + "  ".join(f"{l:>8}" for l in labels)
print(header)

# Rows
for i, label in enumerate(labels):
    row = f"{label:>8} " + "  ".join(f"{tau_mat[i,j]:8.3f}" for j in range(len(labels)))
    print(row)

          ML-Deep   ML-Algo       DNN       NLP
 ML-Deep    1.000     0.600     0.500     0.000
 ML-Algo    0.750     1.000     0.000     0.000
     DNN    0.500     0.000     1.000     0.000
     NLP    0.000     0.000     0.000     1.000


In [ ]:
# Display ρ-matrix (exclusion scores)
rho_mat = matrix['rho_matrix']

print(f"\nρ-matrix (exclusion scores):")
print(header)

for i, label in enumerate(labels):
    row = f"{label:>8} " + "  ".join(f"{rho_mat[i,j]:8.3f}" for j in range(len(labels)))
    print(row)

## 7. Morphism Graph — Categorical Structure

The `morphism_graph()` function builds the directed graph where:
- Nodes are HLLSets
- Edge i → j exists iff BSS_τ(i→j) ≥ τ and BSS_ρ(i→j) ≤ ρ

This reveals the categorical structure of the set collection.

In [19]:
# Build morphism graph with moderate thresholds
graph = morphism_graph(sets, tau_threshold=0.5, rho_threshold=0.6, labels=labels)

print(f"Morphism graph (τ≥0.5, ρ≤0.6):")
print(f"  Nodes: {graph['node_count']}")
print(f"  Edges: {graph['edge_count']}")
print(f"\nEdges:")
for src, tgt, tau, rho in graph['edges']:
    print(f"  {labels[src]} → {labels[tgt]}: τ={tau:.3f}, ρ={rho:.3f}")

Morphism graph (τ≥0.5, ρ≤0.6):
  Nodes: 4
  Edges: 2

Edges:
  ML-Deep → ML-Algo: τ=0.600, ρ=0.400
  ML-Algo → ML-Deep: τ=0.750, ρ=0.500


In [20]:
# Adjacency structure
print(f"\nAdjacency list:")
for node, neighbors in graph['adjacency'].items():
    neighbor_names = [labels[n] for n in neighbors]
    print(f"  {labels[node]} → {neighbor_names if neighbor_names else '(no morphisms)'}")


Adjacency list:
  ML-Deep → ['ML-Algo']
  ML-Algo → ['ML-Deep']
  DNN → (no morphisms)
  NLP → (no morphisms)


In [21]:
# Stricter graph (fewer edges)
strict_graph = morphism_graph(sets, tau_threshold=0.8, rho_threshold=0.2, labels=labels)

print(f"Strict morphism graph (τ≥0.8, ρ≤0.2):")
print(f"  Edges: {strict_graph['edge_count']}")
for src, tgt, tau, rho in strict_graph['edges']:
    print(f"  {labels[src]} → {labels[tgt]}: τ={tau:.3f}, ρ={rho:.3f}")

Strict morphism graph (τ≥0.8, ρ≤0.2):
  Edges: 0


## 8. Practical Example: Document Similarity

Use BSS to find which documents "cover" a query topic.

In [22]:
# Document corpus as HLLSets
docs = {
    "doc1": HLLSet.from_batch("machine learning neural networks deep".split()),
    "doc2": HLLSet.from_batch("natural language processing text mining".split()),
    "doc3": HLLSet.from_batch("computer vision image recognition deep learning".split()),
    "doc4": HLLSet.from_batch("reinforcement learning agents rewards".split()),
}

# Query: "deep learning models"
query = HLLSet.from_batch(["deep", "learning", "models"])

print(f"Query: 'deep learning models'")
print(f"\nDocument coverage (BSS: doc → query):")
for name, doc in docs.items():
    pair = bss(doc, query)
    print(f"  {name}: τ={pair.tau:.3f} (covers {pair.tau*100:.0f}% of query), ρ={pair.rho:.3f}")

Query: 'deep learning models'

Document coverage (BSS: doc → query):
  doc1: τ=0.750 (covers 75% of query), ρ=1.000
  doc2: τ=0.000 (covers 0% of query), ρ=1.000
  doc3: τ=0.750 (covers 75% of query), ρ=1.000
  doc4: τ=0.500 (covers 50% of query), ρ=1.000


In [23]:
# Test morphisms from docs to query
print(f"\nMorphism existence (doc → query, τ≥0.6, ρ≤0.5):")
for name, doc in docs.items():
    result = test_morphism(doc, query, tau_threshold=0.6, rho_threshold=0.5)
    status = "✓" if result.exists else "✗"
    print(f"  {status} {name}: τ={result.tau:.3f}, ρ={result.rho:.3f}")


Morphism existence (doc → query, τ≥0.6, ρ≤0.5):
  ✗ doc1: τ=0.750, ρ=1.000
  ✗ doc2: τ=0.000, ρ=1.000
  ✗ doc3: τ=0.750, ρ=1.000
  ✗ doc4: τ=0.500, ρ=1.000


## 9. BSS for Semantic Overlap

BSS naturally captures semantic relationships when sets represent vocabulary.

In [24]:
# Domain vocabularies
ml_vocab = HLLSet.from_batch([
    "machine", "learning", "model", "training", "prediction",
    "feature", "algorithm", "data", "accuracy", "loss"
])

stats_vocab = HLLSet.from_batch([
    "statistics", "probability", "distribution", "mean", "variance",
    "hypothesis", "regression", "model", "data", "sample"
])

physics_vocab = HLLSet.from_batch([
    "physics", "quantum", "particle", "wave", "energy",
    "momentum", "force", "field", "mass", "velocity"
])

print(f"Domain overlap analysis:")
print(f"\nML → Stats:")
print(bss_summary(ml_vocab, stats_vocab))
print(f"\nStats → ML:")
print(bss_summary(stats_vocab, ml_vocab))
print(f"\nML → Physics:")
print(bss_summary(ml_vocab, physics_vocab))

Domain overlap analysis:

ML → Stats:
BSS(A → B):
  |A| ≈ 12.0,  |B| ≈ 10.0,  |A∩B| ≈ 3.0
  τ (inclusion) = 0.3000
  ρ (exclusion) = 0.9000
  τ + ρ         = 1.2000

Stats → ML:
BSS(A → B):
  |A| ≈ 10.0,  |B| ≈ 12.0,  |A∩B| ≈ 3.0
  τ (inclusion) = 0.2500
  ρ (exclusion) = 0.6667
  τ + ρ         = 0.9167

ML → Physics:
BSS(A → B):
  |A| ≈ 12.0,  |B| ≈ 11.0,  |A∩B| ≈ 0.0
  τ (inclusion) = 0.0000
  ρ (exclusion) = 1.0000
  τ + ρ         = 1.0000
  → A and B are nearly disjoint


## 10. The Directed Nature of BSS

Understanding why direction matters is key to using BSS effectively.

In [25]:
# Scenario: General category vs specific item
category = HLLSet.from_batch(["animal", "mammal", "dog", "cat", "bird", "fish"])
specific = HLLSet.from_batch(["dog", "labrador", "retriever"])

cat_to_spec = bss(category, specific)
spec_to_cat = bss(specific, category)

print(f"General category: {{'animal', 'mammal', 'dog', 'cat', 'bird', 'fish'}}")
print(f"Specific item: {{'dog', 'labrador', 'retriever'}}")
print(f"\nBSS(category → specific): τ={cat_to_spec.tau:.3f}, ρ={cat_to_spec.rho:.3f}")
print(f"  → Category covers {cat_to_spec.tau*100:.0f}% of specific")
print(f"\nBSS(specific → category): τ={spec_to_cat.tau:.3f}, ρ={spec_to_cat.rho:.3f}")
print(f"  → Specific covers {spec_to_cat.tau*100:.0f}% of category")

General category: {'animal', 'mammal', 'dog', 'cat', 'bird', 'fish'}
Specific item: {'dog', 'labrador', 'retriever'}

BSS(category → specific): τ=0.500, ρ=1.000
  → Category covers 50% of specific

BSS(specific → category): τ=0.286, ρ=0.429
  → Specific covers 29% of category


In [26]:
# Interpretation:
# - category → specific: High τ (dog is in category), moderate ρ (extra category items)
# - specific → category: Low τ (specific is small subset), low ρ (most of specific is in category)

print(f"\nThis asymmetry captures:")
print(f"  • The category CONTAINS the specific instance (high τ from cat→spec)")
print(f"  • The specific instance IS CONTAINED BY category (low τ from spec→cat)")


This asymmetry captures:
  • The category CONTAINS the specific instance (high τ from cat→spec)
  • The specific instance IS CONTAINED BY category (low τ from spec→cat)


## Summary

In this tutorial, you learned:

1. **BSS Definition** — τ (inclusion) and ρ (exclusion) scores
2. **Asymmetry** — BSS(A→B) ≠ BSS(B→A) captures directionality
3. **Morphism Testing** — τ ≥ threshold AND ρ ≤ threshold
4. **BSS Matrix** — Pairwise analysis for collections
5. **Morphism Graphs** — Categorical structure discovery

### Key Insights

| Measure | Meaning | High Value | Low Value |
|---------|---------|------------|----------|
| τ (tau) | Inclusion | A covers most of B | A misses most of B |
| ρ (rho) | Exclusion | A has much extraneous content | A is focused on B |

### Morphism Interpretation

- **Strong morphism** (high τ, low ρ): A is a good "summary" of B
- **No morphism** (low τ, high ρ): A and B are unrelated
- **Noisy inclusion** (high τ, high ρ): A covers B but adds noise

### Connection to Other Modules

BSS is used throughout the framework:
- **Disambiguation** (Tutorial 08-12): BSS-based candidate ranking
- **Noether conservation** (Tutorial 11): BSS validates morphism chains
- **Lattice traversal** (Tutorial 03): BSS for temporal similarity

### Next Steps

This completes **Tier 2: Coordination Layer**. Continue to:

- **Tutorial 08**: De Bruijn Graphs — D/R/N transformations
- **Tutorial 09**: HLLSet De Bruijn — Tensor-indexed search
- **Tutorial 10**: Disambiguation — Candidate resolution pipeline